Hola **Guadalupe**!

Soy **Patricio Requena** 👋. Es un placer ser el revisor de tu proyecto el día de hoy!

Revisaré tu proyecto detenidamente con el objetivo de ayudarte a mejorar y perfeccionar tus habilidades. Durante mi revisión, identificaré áreas donde puedas hacer mejoras en tu código, señalando específicamente qué y cómo podrías ajustar para optimizar el rendimiento y la claridad de tu proyecto. Además, es importante para mí destacar los aspectos que has manejado excepcionalmente bien. Reconocer tus fortalezas te ayudará a entender qué técnicas y métodos están funcionando a tu favor y cómo puedes aplicarlos en futuras tareas. 

_**Recuerda que al final de este notebook encontrarás un comentario general de mi parte**_, empecemos!

Encontrarás mis comentarios dentro de cajas verdes, amarillas o rojas, ⚠️ **por favor, no muevas, modifiques o borres mis comentarios** ⚠️:


<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class=“tocSkip”></a>
Si todo está perfecto.
</div>

<div class="alert alert-block alert-warning">
<b>Comentario del revisor</b> <a class=“tocSkip”></a>
Si tu código está bien pero se puede mejorar o hay algún detalle que le hace falta.
</div>

<div class="alert alert-block alert-danger">
<b>Comentario del revisor</b> <a class=“tocSkip”></a>
Si de pronto hace falta algo o existe algún problema con tu código o conclusiones.
</div>

Puedes responderme de esta forma:
<div class="alert alert-block alert-info">
<b>Respuesta del estudiante</b> <a class=“tocSkip”></a>
</div>

# Modelo ML: Pronosticar cancelación de suscripciones

Al operador de telecomunicaciones Interconnect le gustaría poder pronosticar su tasa de cancelación de clientes. Si se descubre que un usuario o usuaria planea irse, se le ofrecerán códigos promocionales y opciones de planes especiales. El equipo de marketing ha recopilado algunos de los datos personales de sus clientes.

Interconnect proporciona principalmente dos tipos de servicios: Comunicación por teléfono fijo e Internet.

Algunos otros servicios que ofrece la empresa incluyen:

- Seguridad en Internet: software antivirus (*DeviceProtection*) y un bloqueador de sitios web maliciosos (*OnlineSecurity*).
- Una línea de soporte técnico (*TechSupport*).
- Almacenamiento de archivos en la nube y backup de datos (*OnlineBackup*).
- Streaming de TV (*StreamingTV*) y directorio de películas (*StreamingMovies*)

La clientela puede elegir entre un pago mensual o firmar un contrato de 1 o 2 años. Puede utilizar varios métodos de pago y recibir una factura electrónica después de una transacción.

Para la predicción, se crean **modelos de clasificación** los cuales se comparan para elegir el mejor.

Como métrica principal para evluar el modelo se usa el valor **AUC-ROC**, es decir el área bajo la curva Característica operativa del receptor *(AUC-ROC: Area under the curve - Receiver Operating Characteristic)*. Esta se usa para medir el acierto en la predicción de eventos binarios.

Como métrica adicional se usa el valor de exactitud

## Tabla de contenido
- Plan de trabajo
- Inicialización
- Preprocesamiento de datos
- Análisis exploratorio de datos (EDA)
 - Correlación de variables
- Entrenamiento y evaluación de modelos
 - Preparar conjuntos de datos
 - Función para evaluar modelos
 - Modelos de regresión logística
 - Modelos de árbol de clasificación
 - Modelos de bosque aleatorio para clasificación
 - Modelos CatBoostClassifier
- Conclusiones

## Plan de trabajo

1. Vista previa y preprocesamiento de datos
2. Hacer un análisis exploratorio de datos (EDA)
 - Incluir correlación de variables

3. Adaptar las posibles características para los entrenamientos de distintos modelos, como transformar las variables categóricas en numéricas segun lo requiera el modelo.

4. Entrenar distintos modelos predictivos con los datos
 - Al separar el conjunto de entrenamiento y prueba, cuidar que el conjunto de entrenamiento tenga datos de contratos cancelados
 - Hacer pruebas con distintos hiperparámetros
 - Tomar como métrica principal el valor AUC-ROC
 - Tomar como métrica adicional la exactitud

5. Comparar los resultados de los modelos

## Inicialización

In [3]:
#instalar catboost
pip install catboost

SyntaxError: invalid syntax (3625326740.py, line 2)

In [11]:
#importar librerias
import pandas as pd
from pandas.api.types import is_numeric_dtype

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier

Se cuenta con 4 archivos:
- `contract.csv` — información del contrato
- `personal.csv` — datos personales del cliente
- `internet.csv` — información sobre los servicios de Internet
- `phone.csv` — información sobre los servicios telefónicos

En cada archivo, la columna `customerID` (ID de cliente) contiene un código único asignado a cada cliente.

In [12]:
#cargar datos
contract = pd.read_csv('https://raw.githubusercontent.com/LupizRmz/Proyecto-Final-/main/contract.csv', parse_dates = [1]))
internet = pd.read_csv('https://raw.githubusercontent.com/LupizRmz/Proyecto-Final-/main/internet.csv')
personal = pd.read_csv('https://raw.githubusercontent.com/LupizRmz/Proyecto-Final-/main/personal.csv')
phone = pd.read_csv('https://raw.githubusercontent.com/LupizRmz/Proyecto-Final-/main/phone.csv')

SyntaxError: unmatched ')' (1159173641.py, line 2)

<div class="alert alert-block alert-danger">
<b>Comentario del revisor (1ra Iteracion)</b> <a class=“tocSkip”></a>

Hay un error de sintaxis en esta celda que carga los datos y al depender el resto del notebook de estas variables no se puede ejecutar en su totalidad.

Cómo práctica siempre antes de compartir un notebook reinicia el kernel del mismo y ejecuta todo desde cero para identificar estos posibles errores antes de compartir tu trabajo.
</div>

In [13]:
#examinar tamaño de dfs
for df in [contract, internet, personal, phone]:
  print(df.shape)

NameError: name 'contract' is not defined

Se oberva que los dfs `contract` y `personal` tienen el mismo número de datos.

La inconsistencia en la cantidad de información/renglones, podría deberse a que no todos los usuarios con plan telefónico tienen plan de internet y viceversa.

**Vista previa de los conjuntos de datos**

In [ ]:
contract.head()

In [ ]:
internet.head()

In [ ]:
personal.head()

In [ ]:
phone.head()

**Información general de los conjuntos de datos**

In [8]:
contract.info()

NameError: name 'contract' is not defined

No hay valores ausentes pero la columna `TotalCharges` tiene el tipo de dato incorrecto.

In [9]:
internet.info()

NameError: name 'internet' is not defined

No hay valores ausentes pero la mayoría de las columnas podría cambiarse a tipo de dato booleano para facilitar la creación del modelo.

In [ ]:
personal.info()

In [ ]:
phone.info()

No hay valores ausentes en ninguno de los conjuntos.

## Preprocesamiento de datos

In [ ]:
#unir todas las tablas
df = contract.merge(internet, how='left', on='customerID')
df = df.merge(personal, how='left', on='customerID')
df = df.merge(phone, how='left', on='customerID')

df.head(3)

Se crea la columna objetivo a partir de la columna `EndDate`. Por ello, esta última no podra ser tomada en cuenta para la construcción del modelo.

In [ ]:
#crear columna objetivo
df['Target'] = (df['EndDate'] != "No").astype("int")
df.head()

In [ ]:
df.info()

**Cambiar tipos de datos erroneos**

In [ ]:
#buscar valor que no es numéricoo
df['TotalCharges'].value_counts().sort_values().tail()

In [ ]:
#explorar filas con valor ausente en columna TotalCharges
df[df['TotalCharges'] == ' ']

In [ ]:
df['BeginDate'].describe(datetime_is_numeric=True)

La cantidad de valores ausentes en la columna `TotalCharges` es mínima y muestra información sobre clientes nuevos por lo que se pueden eliminar estas filas sin afectar el modelo.

In [ ]:
#eliminar renglones con valores ausentes en columna TotalCharges
df = df[df['TotalCharges'] != ' ']

In [ ]:
#cambiar columna TotalCharges a tipo float
df['TotalCharges'] = df['TotalCharges'].astype('float')

In [ ]:
#cambiar columnas valores de 0 y 1
yn_columns = ['PaperlessBilling', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Partner','Dependents', 'MultipleLines']

#asignar 0 a todos los usuarios que no tengan cierto servicio y cambiar tipo de dato
for col in yn_columns:
  df.loc[df[col] == 'Yes', col] = 1
  df.loc[df[col] != 1, col] = 0

  df[col] = df[col].astype('int')

In [ ]:
#verificar cambios
df.info()

Ya no hay valores ausentes

**Explorar columnas categóricas**

In [ ]:
#buscar usuarios duplicados
df['customerID'].duplicated().sum()

In [ ]:
#explorar columnas categóricas
for col in ['Type', 'PaymentMethod', 'InternetService','gender']:
  print(col)
  print(df[col].unique())
  print()

In [ ]:
#rellenar valores ausentes de columna InternetService
df['InternetService'] = df['InternetService'].fillna('None')

#verificar cambios
df['InternetService'].unique()

**Explorar columnas numéricas**

In [ ]:
df[['MonthlyCharges', 'TotalCharges']].describe()

**Explorar columnas booleanas**

In [ ]:
bool_col = ['PaperlessBilling', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'SeniorCitizen', 'Partner', 'Dependents', 'MultipleLines']

for col in bool_col:
  print(df[col].value_counts(normalize=True))
  print()

**Explorar columnas con fechas**

In [ ]:
df['BeginDate'].describe(datetime_is_numeric=True)

In [ ]:
# comprobar que las fechas estan en orden
df['BeginDate'].is_monotonic_increasing

In [ ]:
#ordenar fechas
df = df.sort_values('BeginDate')

In [ ]:
#información de clientes dados de baja
end_date = df[df['EndDate'] != 'No'].copy()
end_date['EndDate'] = pd.to_datetime(end_date['EndDate'])

In [ ]:
#cancelaciones del contrato por fecha
end_date['EndDate'].describe(datetime_is_numeric=True)

Se tienen fechas de inicio de contratos desde el 1 de octubre de 2013, hasta el 1 de enero 2020. Sin embargo, se tienen fechas de cancelaciones desde el 1 de octubre de 2019 hasta el 1 de enero 2020.

Todas las columnas tienen una cantidad significativa de datos en ambas categorías por lo que actualmente ninguna se considera una característica irrelevante para el modelo.

## Análisis exploratorio de datos (EDA)

In [ ]:
#explorar columna objetivo
df['Target'].value_counts(normalize=True)

El 26.58% de los usuarios cancelan el servicio

**Explorar por fecha**

In [ ]:
#configurar BeginDate como indice
df.set_index(keys='BeginDate', inplace=True)

# comprobar que las fechas estan en orden
df.index.is_monotonic_increasing

In [ ]:
#inicializaciones del contrato por mes
df.resample('1M')['Target'].count().plot(figsize=(16,4))
plt.show()

In [ ]:
#inicializaciones del contrato por año
df.resample('1Y')['Target'].count().plot(figsize=(16,4))
plt.show()

Se observa un alto número de inicio de contratos alrededor del 2014. Nuevamente, en el año 2019 incrementa este número.

In [ ]:
#configurar EndDate como indice
end_date.set_index(keys='EndDate', inplace=True)

In [ ]:
#fechas de inicio de contratos cancelados
end_date['BeginDate'].describe(datetime_is_numeric=True)

Al observar únicamente los contratos cancelados, la mitad corresponde a contratos iniciados entre 2013 y 2018 y la otra mitad iniciaron en febrero 2019.

In [ ]:
#cancelaciones por mes
end_date.resample('1M')['Target'].sum().plot(figsize=(16,4))
plt.xlim()
plt.show()

In [ ]:
#cancelaciones por dia
end_date.resample('1D')['Target'].sum().plot(figsize=(16,4))
plt.xlim()
plt.show()

Las cancelaciones iniciaron en octubre del 2019, incrementaron durante noviembre y disminuyeron nuevamente en diciembre.

Solo hay registro de cancelaciones al cambiar el mes, probablemente cerca de la fecha de cobro.

### Correlación de variables - datos históricos

**Correlación de variables numéricas**

In [ ]:
#matriz de dispersión
sns.pairplot(df[['MonthlyCharges', 'TotalCharges', 'Target']],corner=True)
plt.show()

In [ ]:
#matriz de correlación características numéricas 
df[['MonthlyCharges', 'TotalCharges', 'Target']].corr()

Esto muestra que hay una ligera relación positiva entre la columna objetivo y la columna `MonthlyCharges`. Por el contrario, hay una ligera relación negativa entre la columna objetivo y la columna `TotalCharges`.

**Correlación de variables booleanas**

In [ ]:
bool_col.append('Target')

#matriz de correlación características booleanas
df[bool_col].corr()['Target'].sort_values()

Esto muestra que hay una ligera relación negativa entre la columna objetivo y las columnas:
- `OnlineSecurity`
- `TechSupport`
- `Dependents`
- `Partner`

Hay una ligera relación positiva entre la columna objetivo y las columnas:
- `PaperlessBilling`
- `SeniorCitizen`

Hay una relación casi nula (menor a 0.10) entre la columna objetivo y el resto de las columnas.

**Variables categóricas**

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
fig.suptitle('Categorical Variables')

sns.barplot(ax=axes[0], data=df, x='Type', y='Target')
sns.barplot(ax=axes[1], data=df, x='InternetService', y='Target')
sns.barplot(ax=axes[2], data=df, x='gender', y='Target')
sns.barplot(ax=axes[3], data=df, x='PaymentMethod', y='Target')
plt.xticks(rotation=30)
plt.ylim(0,0.5)
plt.show()

En cuanto al **tipo de pago**, los clientes con menor tasa de cancelación son los que pagan anualmente o cada dos años.

En el **servicio de internet**, lo clientes con mayor porcentaje de cancelación son los que cuentan con fibra óptica, seguidos de los que usan una línea telefónica (DSL) y los de menor tasa son los que no contratan el servicio de internet.

La característica **género** no muestra relación con la columna objetivo, en promedio la misma cantidad de hombres y mujeres cancelan los contratos.

En cuanto al **método de pago**, los clientes con mayor tasa de cancelación son los que pagan con cheque electrónico. En los demás no hay una diferencia significativa.

## Entrenamiento y evaluación de modelos

### Preparar conjuntos de datos
En las columnas `PaymentMethod` y `Type` se agrupan las categorías que son similares, en cuanto a la tasa de cancelación.

In [ ]:
#agrupar categorías de columna PaymentMethod
df.loc[df['PaymentMethod'] != 'Electronic check', 'PaymentMethod'] = 'Other'
df['PaymentMethod'].unique()

In [ ]:
#agrupar categorías de columna Type
df.loc[df['Type'] != 'Month-to-month', 'Type'] = '1 or 2 years'
df['Type'].unique()

Se eliminan la columnas que no son relevantes para el entrenamiento del modelo como la columna `customerID`, la columna `gender` que no muestra influencia en el objetivo y la columna `EndDate` que forma parte de la característica objetivo.



In [ ]:
#fijar características para entrenar al modelo
df = df.drop(columns=['customerID', 'EndDate', 'gender'])

Se crean 2 conjuntos de datos, uno histórico y otro con datos únicamente del 2019.
 - Se crean los conjuntos de entrenamiento y prueba
 - Se transforman las características categóricas con OHE

**Datos históricos**

In [ ]:
#separar características de objetivo
features = df.drop(['Target'],axis=1)
target = df['Target']

#separar conjunto de entrenamiento y prueba
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.1, shuffle=False)

**Balancear datos**

Se balancean los datos de entrenamiento al duplicar los datos con cancelaciones pues hay pocas.

In [ ]:
#función para balancear datos
def upsample(features, target):
  repeat = 2

  #Separar respuestas de conjunto de datos de entrenamiento
  features_zeros = features[target == 0]
  features_ones = features[target == 1]
  target_zeros = target[target == 0]
  target_ones = target[target == 1]

  #Aumentar observaciones positivas (donde cancelaron)
  features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
  target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)

  #reordenar datos por fecha
  features_upsampled.sort_index(inplace=True)
  target_upsampled.sort_index(inplace=True)

  return features_upsampled, target_upsampled

In [ ]:
#aumentar datos de cancelaciones
features_train, target_train = upsample(features_train, target_train)

In [ ]:
#comprobar que los datos de entrenamiento son de fechas anteriores a los datos de prueba y que contengan contratos cancelados
print('Training set date range: ', features_train.index.min(),' - ', features_train.index.max())
print('Test set date range: ', features_test.index.min(), ' - ',features_test.index.max())

**Escalar datos**

In [ ]:
#explorar columnas numéricas
df[['MonthlyCharges','TotalCharges']].describe()

Las columnas númericas tienen distintas escalas, los valores mínimos oscilan desde 1 hasta 2013; mientras que los valores máximos oscilan desde 12 hasta 8684.8. Por ello, se estandarizan para darles la misma importancia y así mejorar el modelo.

Primero se escalan todos los datos para más adelante verificar la calidad de los modelos por medio de validación cruzada.

In [ ]:
#escalar todos los datos
features[['MonthlyCharges','TotalCharges']] = StandardScaler().fit_transform(features[['MonthlyCharges','TotalCharges']])

#Verificar cambios
features[['MonthlyCharges','TotalCharges']].describe()

**Datos históricos escalados sin OHE**

In [ ]:
#función para escalar datos
def scale_data(features_train, features_test):
  #establecer escalador y entrenarlo con datos de entrenamiento
  scaler = StandardScaler()
  scaler.fit(features_train[['MonthlyCharges','TotalCharges']])

  #escalar las características de todos los conjuntos de datos
  features_train[['MonthlyCharges','TotalCharges']] = scaler.transform(features_train[['MonthlyCharges','TotalCharges']])
  features_test[['MonthlyCharges','TotalCharges']]  = scaler.transform(features_test[['MonthlyCharges','TotalCharges']])

  #Verificar cambios
  print(features_train[['MonthlyCharges','TotalCharges']].describe())

  return features_train, features_test

In [ ]:
#escalar datos
features_train, features_test = scale_data(features_train, features_test)

**Datos históricos escalados con OHE**

Aplicar codificación One-Hot (OHE) para transformar características categóricas en numéricas y escalar datos.

Nuevamente, se escalan todos los datos para más adelante verificar la calidad de los modelos por medio de validación cruzada.

In [ ]:
#aplicar OHE
features_ohe = pd.get_dummies(features,drop_first=True)

In [ ]:
#escalar todos los datos
features_ohe[['MonthlyCharges','TotalCharges']] = StandardScaler().fit_transform(features_ohe[['MonthlyCharges','TotalCharges']])

#Verificar cambios
features_ohe[['MonthlyCharges','TotalCharges']].describe()

In [ ]:
#separar conjunto de entrenamiento y prueba
features_train_ohe, features_test_ohe, target_train_ohe, target_test_ohe = train_test_split(features_ohe, target, test_size=0.1, shuffle=False)

In [ ]:
#aumentar datos de cancelaciones
features_train_ohe, target_train_ohe = upsample(features_train_ohe, target_train_ohe)

In [ ]:
#comprobar que los datos de entrenamiento son de fechas anteriores a los datos de prueba y que contengan contratos cancelados
print('Training set date range: ', features_train_ohe.index.min(),' - ', features_train_ohe.index.max())
print('Test set date range: ', features_test_ohe.index.min(), ' - ',features_test_ohe.index.max())

In [ ]:
#escalar datos
features_train_ohe, features_test_ohe = scale_data(features_train_ohe, features_test_ohe)

**Datos del año 2019**

Se crean a partir del dataset histórico.

In [ ]:
#crear conjunto de datos del año 2019
df19 = df[df.index>'2018-12-31']

In [ ]:
#separar características de objetivo
features19 = df19.drop(['Target'],axis=1)
target19 = df19['Target']

#separar conjunto de entrenamiento y prueba
features_train19, features_test19, target_train19, target_test19 = train_test_split(features19, target19, test_size=0.2, shuffle=False)

In [ ]:
#aumentar datos de cancelaciones
features_train19, target_train19 = upsample(features_train19, target_train19)

In [ ]:
#comprobar que los datos de entrenamiento son de fechas anteriores a los datos de prueba y que contengan contratos cancelados
print('Training set date range: ', features_train19.index.min(),' - ', features_train19.index.max())
print('Test set date range: ', features_test19.index.min(), ' - ',features_test19.index.max())

**Datos del año 2019 escalados sin OHE**

Nuevamente, se escalan todos los datos para más adelante verificar la calidad de los modelos por medio de validación cruzada.

In [ ]:
#escalar todos los datos
features19[['MonthlyCharges','TotalCharges']] = StandardScaler().fit_transform(features19[['MonthlyCharges','TotalCharges']])

#Verificar cambios
features19[['MonthlyCharges','TotalCharges']].describe()

In [ ]:
#escalar datos
features_train19, features_test19 = scale_data(features_train19, features_test19)

**Datos del año 2019 escalados con OHE**

In [ ]:
features_ohe19 = pd.get_dummies(features19,drop_first=True)

#separar conjunto de entrenamiento y prueba
features_train_ohe19, features_test_ohe19, target_train_ohe19, target_test_ohe19 = train_test_split(features_ohe19, target19, test_size=0.2, shuffle=False)

In [ ]:
#aumentar datos de cancelaciones
features_train_ohe19, target_train_ohe19 = upsample(features_train_ohe19, target_train_ohe19)

In [ ]:
#comprobar que los datos de entrenamiento son de fechas anteriores a los datos de prueba y que contengan contratos cancelados
print('Training set date range: ', features_train_ohe19.index.min(),' - ', features_train_ohe19.index.max())
print('Test set date range: ', features_test_ohe19.index.min(), ' - ',features_test_ohe19.index.max())

In [ ]:
#escalar todos los datos
features_ohe19[['MonthlyCharges','TotalCharges']] = StandardScaler().fit_transform(features_ohe19[['MonthlyCharges','TotalCharges']])

#Verificar cambios
features_ohe19[['MonthlyCharges','TotalCharges']].describe()

In [ ]:
#escalar datos
features_train_ohe19, features_test_ohe19 = scale_data(features_train_ohe19, features_test_ohe19)

### Función para evaluar modelos

Se crea una función para graficar la métrica AUC-ROC, obtener su valor y para obtener la exactitud de los modelos.

In [ ]:
def metrics_plot(all_features, all_targets, features_train, features_test, target_train, target_test, model):
  #graficar AUC-ROC conjunto de entrenamiento
  fpr1, tpr1, thresholds1 = roc_curve(target_train, model.predict_proba(features_train)[:, 1])

  #graficar AUC-ROC conjunto de prueba
  fpr2, tpr2, thresholds2 = roc_curve(target_test, model.predict_proba(features_test)[:, 1])

  plt.figure()
  plt.plot(fpr1, tpr1, color='b', label='Training set')
  plt.plot(fpr2, tpr2, color='c', label='Test set')
  # Curva ROC para modelo aleatorio (línea recta)
  plt.plot([0, 1], [0, 1], linestyle='--', color='orange', label='Random model')

  #Establecer límite para los ejes de 0 a 1
  plt.ylim([0.0, 1.0])
  plt.xlim([0.0, 1.0])

  #nombrar ejes y encabezado
  plt.legend()
  plt.xlabel("False positive rate")
  plt.ylabel("True positive rate")
  plt.title("ROC Curve")

  plt.show()

  #métrica AUC-ROC
  roc_auc_train = roc_auc_score(target_train, model.predict_proba(features_train)[:, 1])
  roc_auc_test = roc_auc_score(target_test, model.predict_proba(features_test)[:, 1])

  print()
  print('ROC-AUC Score - train set: ', roc_auc_train)
  print('ROC-AUC Score - test set: ', roc_auc_test)

  #calcular métrica exactitud
  acc_train = accuracy_score(target_train, model.predict(features_train))
  acc_test = accuracy_score(target_test, model.predict(features_test))

  print()
  print('Accuracy - train set: ', acc_train)
  print('Accuracy - test set: ', acc_test)

  #verificar calidad del modelo con validación cruzada
  print()
  print('Cross validation score:',( cross_val_score(model, all_features, all_targets, scoring='roc_auc', cv=5) ).mean())

  return roc_auc_train, roc_auc_test, acc_train, acc_test

### Modelo de regresión logística

**Datos históricos**

Se usan los datos históricos con OHE.

In [ ]:
#establecer modelo y entrenarlo
model_lr = LogisticRegression(solver='newton-cholesky')
model_lr.fit(features_train_ohe, target_train_ohe)

In [ ]:
#evaluar modelo
roc_auc_lr_train, roc_auc_lr_test, acc_lr_train, acc_lr_test = metrics_plot(features_ohe, target, features_train_ohe, features_test_ohe,
                                                                            target_train_ohe, target_test_ohe, model_lr)

Se observa una buena calidad del modelo y no hay sobreajuste.

**Datos del año 2019**

Se usan los datos del año 2019 con OHE.

In [ ]:
#establecer modelo y entrenarlo
model_lr19 = LogisticRegression(solver='newton-cholesky')

model_lr19.fit(features_train_ohe19, target_train_ohe19)

In [ ]:
#evaluar modelo
roc_auc_lr_train19, roc_auc_lr_test19, acc_lr_train19, acc_lr_test19 = metrics_plot(features_ohe, target, features_train_ohe19,
                                                                                    features_test_ohe19, target_train_ohe19, target_test_ohe19, model_lr19)


**Modelo de regresión logística con uso de descenso de gradiente estocástico - Datos históricos**

El modelo *`SGDClassifier`* no cuenta con el calculo de probabilidades, se usa junto con *`CalibratedClassifierCV`* para calcularlas y poder obtener la métrica *`AUC-ROC`*.

In [ ]:
#establecer modelo y entrenarlo
model_sgd = SGDClassifier(loss='log_loss')
model_sgd_fitted =  model_sgd.fit(features_train_ohe, target_train_ohe)

model_sgd_cal = CalibratedClassifierCV(model_sgd_fitted)
model_sgd_cal.fit(features_train_ohe, target_train_ohe)

In [ ]:
#evaluar modelo
roc_auc_sgd_train, roc_auc_sgd_test, acc_sgd_train, acc_sgd_test = metrics_plot(features_ohe, target, features_train_ohe,
                                                                                features_test_ohe, target_train_ohe, target_test_ohe, model_sgd_cal)

**Modelo de regresión logística con uso de descenso de gradiente estocástico - Datos del año 2019**

In [ ]:
#establecer modelo y entrenarlo
model_sgd19 = SGDClassifier()
model_sgd_fitted19 =  model_sgd19.fit(features_train_ohe19, target_train_ohe19)

model_sgd_cal19 = CalibratedClassifierCV(model_sgd_fitted19)
model_sgd_cal19.fit(features_train_ohe19, target_train_ohe19)

In [ ]:
#evaluar modelo
roc_auc_sgd_train19, roc_auc_sgd_test19, acc_sgd_train19, acc_sgd_test19 = metrics_plot(features_ohe19, target19, features_train_ohe19, features_test_ohe19,
                                                                                        target_train_ohe19, target_test_ohe19, model_sgd_cal19)

### Modelo árbol de clasificación

**Datos históricos**

Se usan los datos históricos con OHE.

In [ ]:
#usar GridSearchCV para encontrar la mejor profundidad de árbol
param_grid = {'max_depth': [1,2,4,6,8,10,15,20]}

tree = DecisionTreeClassifier(random_state=12345)
grid_search_dt = GridSearchCV(estimator=tree, param_grid=param_grid, verbose=True, scoring='roc_auc')

grid_search_dt.fit(features_train_ohe, target_train_ohe)
grid_search_dt.best_estimator_

Las características anteriores se toman como referencia. Se modifica el modelo para evitar el sobreajuste.

In [ ]:
#establecer modelo y entrenarlo
model_dt = DecisionTreeClassifier(max_depth=4, random_state=12345)
model_dt.fit(features_train_ohe, target_train_ohe)

In [ ]:
#evaluar modelo
roc_auc_dt_train, roc_auc_dt_test, acc_dt_train, acc_dt_test = metrics_plot(features_ohe, target, features_train_ohe, features_test_ohe,
                                                                            target_train_ohe, target_test_ohe, model_dt)



**Datos del año 2019**

Se usan los datos del año 2019 con OHE.

In [ ]:
#usar GridSearchCV para encontrar la mejor profundidad de árbol
param_grid = {'max_depth': [1,2,4,6,8,10,15,20]}

tree19 = DecisionTreeClassifier(random_state=12345)
grid_search_dt19 = GridSearchCV(estimator=tree19, param_grid=param_grid, verbose=True, scoring='roc_auc')

grid_search_dt19.fit(features_train_ohe19, target_train_ohe19)
grid_search_dt19.best_estimator_

Las características anteriores se toman como referencia. Se modifica el modelo para evitar el sobreajuste.

In [ ]:
#establecer modelo y entrenarlo
model_dt19 = DecisionTreeClassifier(max_depth=3, random_state=12345)
model_dt19.fit(features_train_ohe19, target_train_ohe19)

In [ ]:
#evaluar modelo
roc_auc_dt_train19, roc_auc_dt_test19, acc_dt_train19, acc_dt_test19 = metrics_plot(features_ohe19, target19, features_train_ohe19,
                                                                                    features_test_ohe19, target_train_ohe19, target_test_ohe19, model_dt19)

### Modelo bosque aleatorio para clasificación

**Datos históricos**

Se usan los datos históricos con OHE.

In [ ]:
#buscar las mejores características para los datos disponibles
param_grid = {'n_estimators': [1,2,3,4,6,8,10,20],'max_features': [ 'sqrt', 'log2'],'max_depth' : [2,3,4,5,6,8,10,20]}

model_rfc = RandomForestClassifier(random_state=12345)
grid_search_rfc = GridSearchCV(estimator=model_rfc, param_grid=param_grid, scoring='roc_auc')

grid_search_rfc.fit(features_train_ohe, target_train_ohe)
grid_search_rfc.best_params_

Las características anteriores se toman como referencia. Se modifica el modelo para evitar el sobreajuste.

In [ ]:
#establecer modelo y entrenarlo
model_rfc = RandomForestClassifier(max_depth=3, max_features='sqrt', n_estimators=6, random_state=12345)
model_rfc.fit(features_train_ohe, target_train_ohe)

In [ ]:
#evaluar modelo
roc_auc_rcf_train, roc_auc_rcf_test, acc_rcf_train, acc_rcf_test = metrics_plot(features_ohe, target, features_train_ohe, features_test_ohe,
                                                                                target_train_ohe, target_test_ohe, model_rfc)

**Datos del año 2019**

Se usan los datos del año 2019 con OHE.

In [ ]:
#buscar las mejores características para los datos disponibles
param_grid = {'n_estimators': [1,2,3,4,6,8,10,20],'max_features': [ 'sqrt', 'log2'],'max_depth' : [2,3,4,5,6,8,10,20]}

model_rfc19 = RandomForestClassifier(random_state=12345)
grid_search_rfc19 = GridSearchCV(estimator=model_rfc19, param_grid=param_grid, scoring='roc_auc')

grid_search_rfc19.fit(features_train_ohe19, target_train_ohe19)
grid_search_rfc19.best_params_

Las características anteriores se toman como referencia. Se modifica el modelo para evitar el sobreajuste.

In [ ]:
#establecer modelo y entrenarlo
model_rfc19 = RandomForestClassifier(max_depth=5, max_features='sqrt', n_estimators=20, random_state=12345)
model_rfc19.fit(features_train_ohe19, target_train_ohe19)

In [ ]:
#evaluar modelo
roc_auc_rcf_train19, roc_auc_rcf_test19, acc_rcf_train19, acc_rcf_test19 = metrics_plot(features_ohe19, target19, features_train_ohe19,
                                                                                        features_test_ohe19, target_train_ohe19, target_test_ohe19, model_rfc19)

### Modelo CatBoostClassifier

**Datos históricos**

Se usan los datos históricos sin OHE.

In [ ]:
#explorar tipos de datos
features_train.info()

In [ ]:
#identificar datos categoricos
train_data = Pool(data = features_train, label = target_train, cat_features = [0,2,5])
test_data =  Pool(data = features_test, label = target_test, cat_features = [0,2,5])

In [ ]:
#transformar características categóricas de todo el conjunto de datos y del entrenamiento por separado
features_cb = features.apply(LabelEncoder().fit_transform)

features_train_cb = features_train.apply(LabelEncoder().fit_transform)

In [5]:
#buscar las mejores características
param_cb={'depth':[4, 6, 8, 10], 'learning_rate':[0.5, 0.1, 0.01], 'iterations':[10, 30, 50, 100]}

model_cb=CatBoostClassifier(random_seed=123456, verbose=False)
grid_cb=GridSearchCV(estimator=model_cb, param_grid=param_cb, n_jobs=-1, scoring='roc_auc')

grid_cb.fit(features_train_cb, target_train)
grid_cb.best_params_

NameError: name 'CatBoostClassifier' is not defined

Las características anteriores se toman como referencia. Se modifica el modelo para evitar el sobreajuste.

In [ ]:
#establecer modelo y entrenarlo
model_cb = CatBoostClassifier(depth=3, iterations=70, learning_rate=0.05, random_seed=123456, verbose=False)
model_cb.fit(train_data)

In [ ]:
#evaluar modelo
roc_auc_cb_train, roc_auc_cb_test, acc_cb_train, acc_cb_test = metrics_plot(features_cb, target, train_data, test_data, target_train, target_test, model_cb)

**Datos del año 2019**

Se usan los datos del año 2019 sin OHE.

In [ ]:
#identificar datos categoricos
train_data19 = Pool(data = features_train19, label = target_train19, cat_features = [0,2,5])
test_data19 =  Pool(data = features_test19, label = target_test19, cat_features = [0,2,5])

In [ ]:
#transformar características categóricas de todo el conjunto de datos y del entrenamiento por separado
features_cb19 = features19.apply(LabelEncoder().fit_transform)

features_train_cb19 = features_train19.apply(LabelEncoder().fit_transform)

In [ ]:
#buscar las mejores características
param_cb={'depth':[4, 6, 8, 10], 'learning_rate':[0.5, 0.1, 0.01], 'iterations':[10, 30, 50, 100]}

model_cb19 = CatBoostClassifier(random_seed=123456, verbose=False)
grid_cb19 = GridSearchCV(estimator=model_cb19, param_grid=param_cb, n_jobs=-1, scoring='roc_auc')

grid_cb19.fit(features_train_cb19, target_train19)
grid_cb19.best_params_

Las características anteriores se toman como referencia. Se modifica el modelo para evitar el sobreajuste.

In [ ]:
#establecer modelo y entrenarlo
model_cb19 = CatBoostClassifier(depth=6, iterations=10, learning_rate=0.4, random_seed=123456, verbose=False)
model_cb19.fit(train_data19)

In [ ]:
#evaluar modelo
roc_auc_cb_train19, roc_auc_cb_test19, acc_cb_train19, acc_cb_test19 = metrics_plot(features_cb19, target19, train_data19, test_data19,
                                                                                    target_train19, target_test19, model_cb19)

## Conclusiones

**Comparación de modelos - datos históricos**

In [ ]:
#Tabla con los valores AUC-ROC y exactitud de los modelos
table1 = pd.DataFrame({     'Model':['LogisticRegression', 'DecisionTreeClassifier', 'RandomForestClassifier', 'CatBoostClassifier'],
       'ROC-AUC Score Training set':[roc_auc_lr_train, roc_auc_dt_train, roc_auc_rcf_train, roc_auc_cb_train],
       'ROC-AUC Score Test set'    :[roc_auc_lr_test,  roc_auc_dt_test,  roc_auc_rcf_test,  roc_auc_cb_test],
       'Accuracy Training set'     :[acc_lr_train,     acc_dt_train,     acc_rcf_train,     acc_cb_train],
       'Accuracy Test set'         :[acc_lr_test,      acc_dt_test,      acc_rcf_test,      acc_cb_test]})

table1.sort_values('ROC-AUC Score Test set', ascending=False)

**Comparación de modelos - datos del año 2019**

In [ ]:
#Tabla con los valores AUC-ROC y exactitud de los modelos
table1 = pd.DataFrame({     'Model':['LogisticRegression', 'DecisionTreeClassifier', 'RandomForestClassifier', 'CatBoostClassifier'],
       'ROC-AUC Score Training set':[roc_auc_lr_train19, roc_auc_dt_train19, roc_auc_rcf_train19, roc_auc_cb_train19],
       'ROC-AUC Score Test set'    :[roc_auc_lr_test19,  roc_auc_dt_test19,  roc_auc_rcf_test19,  roc_auc_cb_test19],
       'Accuracy Training set'     :[acc_lr_train19,     acc_dt_train19,     acc_rcf_train19,     acc_cb_train19],
       'Accuracy Test set'         :[acc_lr_test19,      acc_dt_test19,      acc_rcf_test19,      acc_cb_test19]})

table1.sort_values('ROC-AUC Score Test set', ascending=False)

Al comparar los modelos, en ambas tablas tienen el mismo orden de exito y el modelo **CatBoostClassifier** tiene el `ROC-AUC Score` más alto en el conjunto de prueba.

Tomando como métrica el valor `ROC-AUC Score`, comparando el modelo que toma los datos históricos contra los datos del año 2019, este último conjunto da un `ROC-AUC Score` ligeramente mayor al de los datos históricos. Sin embargo, al tomar los datos históricos se obtiene un valor de `exactitud` (`Accuracy`) significativamente mayor en el conjunto de datos de prueba. Por ello, se recomienda usar el modelo **CatBoostClassifier** con los datos históricos.

En general, los resultados `ROC-AUC Score` de los modelos que solo tomaron los datos del 2019 fueron mejores que los que usaron datos históricos cosa que podría significar que falta información y es recomendable recabar más datos.

